In [ ]:
from collections import defaultdict, Counter
import uuid

In [ ]:

class SemanticCache:
    """
    A simple semantic cache implementation using a dictionary to store query-answer pairs.
    """
    def __init__(self, uuid, threshold=0.8):
        self.uuid = uuid
        self.cache = {self.uuid: defaultdict()}
        self.threshold = threshold

    def _cosine_similarity(self, vec1, vec2):
        """Calculate the cosine similarity between two vectors."""
        dot_product = sum(a * b for a, b in zip(vec1, vec2))
        magnitude_vec1 = sum(a ** 2 for a in vec1) ** 0.5
        magnitude_vec2 = sum(b ** 2 for b in vec2) ** 0.5
        if magnitude_vec1 == 0 or magnitude_vec2 == 0:
            return 0.0
        return dot_product / (magnitude_vec1 * magnitude_vec2)

    def add_query_answer_to_cache(self, query, answer):
        self.cache[self.uuid][query] = answer
        print(f"Added to cache for user {self.uuid}: '{query}' -> '{answer}'")

    def _get_vocab(self, query, cache_store):
        cached_queries = list(cache_store.keys())
        vocab  = list(set(query.lower().split() + [w for q in cached_queries for w in q.lower().split()]))
        return vocab
    
    def _encode(self, word_count, vocab):
        return [word_count.get(word, 0) for word in vocab]
    
    def _vectorize(self, text, vocab):
        word_count = Counter(text.lower().split())
        encoded_sentence = self._encode(word_count, vocab)
        return encoded_sentence

    def get_cache(self):
        user_cache = self.cache.get(self.uuid, {})
        return user_cache

    def get_answer_from_cache(self, query):
        cache_store = self.get_cache()
        vocab = self._get_vocab(query, cache_store)
        query_vector = self._vectorize(query, vocab)
        list_query_vector = [query_vector] * len(cache_store)
        list_vect_cache_queries = [self._vectorize(cached_query, vocab) for cached_query in cache_store.keys()]

        scores = list(map(lambda q, q_cache: self._cosine_similarity(q, q_cache), 
                          [q for q in list_query_vector],
                          [q_cache for q_cache in list_vect_cache_queries]))
        
        best_score_index = scores.index(max(scores))
        if scores[best_score_index] >= self.threshold:
            best_matching_query = list(cache_store.keys())[best_score_index]
            return cache_store[best_matching_query]
        return "No similar query found in cache."

In [ ]:
THRESHOLD = 0.8

semantic_cache = SemanticCache(uuid=str(uuid.uuid4()), threshold=THRESHOLD)

semantic_cache.add_query_answer_to_cache(query='What is the capital of France?', 
                                         answer='The capital of France is Paris.')

semantic_cache.add_query_answer_to_cache(query='What is the largest mammal?',
                                         answer='The largest mammal is the blue whale.')

In [ ]:
cached_answer = semantic_cache.get_answer_from_cache('What is the capital of France?')

if cached_answer:
    print(f"Cached answer: {cached_answer}")
else:
    print("No suitable cached answer found.")

In [ ]:
cached_answer = semantic_cache.get_answer_from_cache('Where are the Eiffel Tower and the Louvre located?')

if cached_answer:
    print(f"Cached answer: {cached_answer}")
else:
    print("No suitable cached answer found.")

In [ ]:
# add a new query-answer pair to the cache
_ = semantic_cache.add_query_answer_to_cache(query='Where are the Eiffel Tower and the Louvre located?',
                                         answer='The Eiffel Tower and the Louvre are located in Paris, France.')


In [ ]:
semantic_cache.get_cache()